# Machine Learning Utilities (`ml_utilities`)

El módulo `ml_utilities` (`ds_utils.ml.ml_utilities`) proporciona funciones de soporte y utilidades para flujos de trabajo de Machine Learning, con especial énfasis en:
- **Detección y explicación de anomalías**: generación de diagnósticos e interpretabilidad para observaciones anómalas en series temporales.
- **Normalización de puntuaciones de anomalía**: escalado robusto (Min-Max invertido acotado por percentiles) y estandarización por rangos percentilares para homogeneizar outputs de modelos no supervisados.
- **Gestión e inspección de modelos**: formateo legible de hiperparámetros (p. ej. `max_features`), filtrado de configuraciones explícitas de usuario y normalización de nombres de algoritmos a identificadores estándar `snake_case`.

In [14]:
# RECARGA AUTOMÁTICA DE MÓDULOS EXTERNOS
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

# Dependencia opcional del extra EDA plot_utilities
try:
    from sklearn.inspection import permutation_importance
except ImportError as exc:
    raise ImportError(
        "This notebook requires the ML dependencies. "
        "Install them with: uv sync --extra ml"
    ) from exc

from ds_utils.ml.ml_utilities import (
    normalize_model_name,
    format_max_features,
    extract_used_params,
    explain_anomalies,
    normalize_anomaly_scores_min_max,
    normalize_anomaly_scores_rank,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## `normalize_model_name`

- **Qué hace**: Transforma el nombre textual de un modelo o algoritmo (que puede incluir mayúsculas, espacios, guiones o signos de puntuación) en un identificador limpio y reproducible en formato `snake_case`. Permite nombrar de manera uniforme columnas en DataFrames de resultados, claves de diccionarios, carpetas o archivos de artefactos.
- **Parámetros**:
  - `model_name` (`str`): Nombre del modelo a normalizar.
- **Qué devuelve**:
  - `str`: Identificador normalizado en minúsculas y separado por guiones bajos.

In [15]:
# Normalización de diferentes nomenclaturas de modelos
sample_models = [
    "Isolation Forest",
    "Local Outlier Factor (LOF)",
    "One-Class SVM",
    "Elliptic Envelope (Robust Covariance)",
    "AutoEncoder -- Deep Learning",
]

normalized_names = [normalize_model_name(name) for name in sample_models]

df_names = pd.DataFrame({
    "nombre_original": sample_models,
    "identificador_normalizado": normalized_names
})
display(df_names)

,nombre_original,identificador_normalizado
0,Isolation Forest,isolation_forest
1,Local Outlier Factor (LOF),local_outlier_factor_lof
2,One-Class SVM,one_class_svm
3,Elliptic Envelope (Robust Covariance),elliptic_envelope_robust_covariance
4,AutoEncoder -- Deep Learning,autoencoder_deep_learning


## `format_max_features`

- **Qué hace**: Convierte los diferentes valores que puede tomar el parámetro `max_features` en modelos basados en árboles (como los de scikit-learn: `"sqrt"`, `"log2"`, un entero, un float fraccionario o `None`) en una descripción textual clara que indica tanto el criterio como la proporción de variables que representa respecto al número total de características del problema.
- **Parámetros**:
  - `max_features` (`str | float | int | None`): Valor del parámetro `max_features` configurado en el modelo.
  - `n_features` (`int`): Número total de columnas/features en el conjunto de entrenamiento.
- **Qué devuelve**:
  - `str`: Texto legible como `"sqrt (0.20)"`, `"log2 (0.16)"`, `"fraction (0.5)"` o `"all (1.00)"`.

In [16]:
# Total de variables disponibles en un problema de telemetría
total_features = 25

configurations = ["sqrt", "log2", None, 0.5, 10]
results = []

for cfg in configurations:
    formatted = format_max_features(max_features=cfg, n_features=total_features)
    results.append({
        "max_features_param": str(cfg),
        "total_features": total_features,
        "representacion_legible": formatted
    })

display(pd.DataFrame(results))

,max_features_param,total_features,representacion_legible
0,sqrt,25,sqrt (0.20)
1,log2,25,log2 (0.19)
2,None,25,all (1.00)
3,0.5,25,fraction (0.5)
4,10,25,fraction (10)


## `extract_used_params`

- **Qué hace**: Toma un diccionario completo de parámetros de un modelo (como el devuelto por `model.get_params()`) y filtra únicamente aquellos hiperparámetros que fueron explícitamente especificados o seleccionados por el usuario. Es útil para auditar o guardar en metadatos solo las configuraciones personalizadas sin arrastrar decenas de valores por defecto.
- **Parámetros**:
  - `all_params` (`dict`): Diccionario con la totalidad de los parámetros del modelo.
  - `used_params` (`dict`): Diccionario que define los parámetros explícitos de interés.
- **Qué devuelve**:
  - `dict`: Subconjunto de `all_params` que contiene solo las claves coincidentes con `used_params`.

In [17]:
# Parámetros totales típicos de un estimador
all_estimator_params = {
    "n_estimators": 100,
    "max_samples": "auto",
    "contamination": 0.05,
    "max_features": 1.0,
    "bootstrap": False,
    "n_jobs": -1,
    "random_state": 42,
    "verbose": 0,
    "warm_start": False,
}

# Hiperparámetros modificados o pedidos por el usuario en su experimento
user_requested_params = {
    "n_estimators": 200,
    "contamination": 0.08,
    "random_state": 42,
    "non_existing_key": "ignored",  # clave que no pertenece al estimador
}

extracted = extract_used_params(
    all_params=all_estimator_params,
    used_params=user_requested_params
)

print("Parámetros filtrados coincidentes:")
display(extracted)

Parámetros filtrados coincidentes:


{'n_estimators': 100, 'contamination': 0.05, 'random_state': 42}

## `explain_anomalies`

- **Qué hace**: Genera diagnósticos comprensibles para personas sobre qué factores causaron cada anomalía detectada en una serie temporal multivariante. Compara el valor actual de cada variable numérica de la fila anómala contra su mediana histórica y su percentil 99, clasificando las variables según su desviación absoluta. Genera tanto un resumen en texto como una lista estructurada con los valores observados.
- **Parámetros**:
  - `df` (`pd.DataFrame`): DataFrame con los datos temporales y las variables numéricas analizadas.
  - `col_time` (`str`): Nombre de la columna que contiene la marca de tiempo (`timestamp`).
  - `anomaly_scores` (`ArrayLike`): Puntuaciones de anomalía devueltas por el detector (un score por fila).
  - `anomaly_mask` (`ArrayLike`): Máscara booleana donde `True` identifica una anomalía confirmada y `False` una observación normal.
  - `top_features` (`int`, por defecto `5`): Número máximo de variables con mayor desviación a incluir en la explicación.
  - `ascending` (`bool`, por defecto `True`): Dirección de ordenamiento del resultado respecto al anomaly score.
- **Qué devuelve**:
  - `pd.DataFrame`: Tabla con una fila por cada anomalía detectada, con las columnas `timestamp`, `score`, `summary` y `details`.

In [18]:
# 1. Generación de telemetría sintética con 2 anomalías simuladas
np.random.seed(42)
n_points = 25
time_series = pd.date_range("2024-01-01 08:00:00", periods=n_points, freq="15min")

df_telemetry = pd.DataFrame({
    "timestamp": time_series,
    "temperature": np.random.normal(loc=22.0, scale=0.4, size=n_points),
    "vibration": np.random.normal(loc=1.1, scale=0.08, size=n_points),
    "pressure": np.random.normal(loc=101.3, scale=0.5, size=n_points),
})

# Inyectamos dos anomalías evidentes en índices concretos
df_telemetry.loc[8, "temperature"] = 58.5   # Sobrecalentamiento crítico
df_telemetry.loc[18, "vibration"] = 9.2     # Vibración anómala mecánica

# Puntuaciones simuladas (en este modelo scores muy negativos representan anomalías severas)
anomaly_scores = np.random.uniform(0.1, 0.4, size=n_points)
anomaly_scores[8] = -0.72
anomaly_scores[18] = -0.61

# Máscara booleana de anomalías
anomaly_mask = anomaly_scores < 0.0

# 2. Llamada a explain_anomalies
df_explanations = explain_anomalies(
    df=df_telemetry,
    col_time="timestamp",
    anomaly_scores=anomaly_scores,
    anomaly_mask=anomaly_mask,
    top_features=3,
    ascending=True,
)

print("Resumen diagnóstico de anomalías:")
display(df_explanations[["timestamp", "score", "summary"]])

print("\nDetalle cuantitativo de la primera anomalía:")
display(pd.DataFrame(df_explanations.loc[0, "details"]))

Resumen diagnóstico de anomalías:


,timestamp,score,summary
0,2024-01-01 10:00:00,-0.72,"temperature (actual=58.500, mediana=21.910, P9..."
1,2024-01-01 12:30:00,-0.61,"vibration (actual=9.200, mediana=1.091, P99=7...."



Detalle cuantitativo de la primera anomalía:


,feature,current,median,p99,deviation
0,temperature,58.500000,21.909689,49.891604,36.590311
1,vibration,1.015383,1.090748,7.291564,0.075365
2,pressure,101.465632,101.462042,102.079129,0.003590


## `normalize_anomaly_scores_min_max`

- **Qué hace**: Estandariza puntuaciones de anomalía en el intervalo acotado `[0, 1]` mediante un reescalado Min-Max invertido y resistente a outliers extremos. La convención aplicada es que `0` representa la observación más normal y `1` la observación más anómala. En lugar de usar los valores mínimos y máximos absolutos, calcula los límites en base a percentiles configurables (`lower_percentile` y `upper_percentile`), recortando los valores fuera de rango para evitar distorsiones por valores atípicos aislados.
- **Parámetros**:
  - `scores` (`pd.Series | pd.DataFrame`): Puntuaciones brutas devueltas por el detector de anomalías (donde menor puntuación implica mayor severidad).
  - `lower_percentile` (`float`, por defecto `0.01`): Percentil inferior del escalado.
  - `upper_percentile` (`float`, por defecto `0.99`): Percentil superior del escalado.
- **Qué devuelve**:
  - `pd.Series | pd.DataFrame`: Puntuaciones normalizadas en `[0, 1]`, preservando el tipo y las dimensiones de entrada.

In [19]:
# Convertimos las puntuaciones del ejemplo anterior en una Serie de Pandas
series_scores = pd.Series(anomaly_scores, name="raw_anomaly_score")

# Normalización Min-Max robusta
scores_min_max = normalize_anomaly_scores_min_max(
    scores=series_scores,
    lower_percentile=0.01,
    upper_percentile=0.99
)

df_comparison_min_max = pd.DataFrame({
    "timestamp": df_telemetry["timestamp"],
    "raw_score": series_scores,
    "min_max_normalized": scores_min_max
})

print("Top observaciones más anómalas (normalizadas próximas a 1.0):")
display(df_comparison_min_max.sort_values(by="min_max_normalized", ascending=False).head(5))

Top observaciones más anómalas (normalizadas próximas a 1.0):


,timestamp,raw_score,min_max_normalized
8,2024-01-01 10:00:00,-0.720000,1.000000
18,2024-01-01 12:30:00,-0.610000,0.921936
0,2024-01-01 08:00:00,0.107626,0.251831
2,2024-01-01 08:30:00,0.109429,0.250148
11,2024-01-01 10:45:00,0.123094,0.237387


## `normalize_anomaly_scores_rank`

- **Qué hace**: Normaliza las puntuaciones de anomalía utilizando la posición de ranking percentilar (`1 - scores.rank(pct=True)`). Transforma las puntuaciones en una escala homogénea en `[0, 1]`, donde `0` es el registro menos anómalo y `1` el más severo. Preserva el orden relativo estricto entre observaciones y resulta especialmente útil para comparar o combinar directamente predicciones de múltiples modelos con funciones de score incompatibles.
- **Parámetros**:
  - `scores` (`pd.Series | pd.DataFrame`): Puntuaciones brutas de anomalía a clasificar.
- **Qué devuelve**:
  - `pd.Series | pd.DataFrame`: Puntuaciones normalizadas por rango en el intervalo `[0, 1]`.

In [20]:
# Normalización por rango percentilar
scores_rank = normalize_anomaly_scores_rank(scores=series_scores)

# Comparativa lado a lado entre puntuación bruta, Min-Max y Ranking
df_all_norm = pd.DataFrame({
    "timestamp": df_telemetry["timestamp"],
    "score_bruto": series_scores,
    "norm_min_max": scores_min_max,
    "norm_rank": scores_rank,
})

print("Comparativa completa de normalización (ordenada por severidad de anomalía):")
display(df_all_norm.sort_values(by="norm_rank", ascending=False).head(6))

Comparativa completa de normalización (ordenada por severidad de anomalía):


,timestamp,score_bruto,norm_min_max,norm_rank
8,2024-01-01 10:00:00,-0.720000,1.000000,0.96
18,2024-01-01 12:30:00,-0.610000,0.921936,0.92
0,2024-01-01 08:00:00,0.107626,0.251831,0.88
2,2024-01-01 08:30:00,0.109429,0.250148,0.84
11,2024-01-01 10:45:00,0.123094,0.237387,0.80
1,2024-01-01 08:15:00,0.132367,0.228728,0.76
